# 04 – Feature Engineering: Variables de Empleo

**Proyecto:** Predicción de Subempleo por Insuficiencia de Horas — EPEN 2024  
**Etapa:** Feature Engineering – Paso 3 de 4  
**Dataset de entrada:** `data/feature_engineering/epen_fe_education.csv`  
**Dataset de salida:** `data/feature_engineering/epen_fe_employment.csv`

## Objetivo

Crear variables derivadas de las **condiciones laborales** del trabajador: categoría ocupacional, sector de empleo, formalidad, tamaño de empresa, búsqueda de otro empleo, horas trabajadas y protección social.

Las variables de empleo son las **más directamente relacionadas** con el subempleo por insuficiencia de horas.

### Variables base utilizadas
| Variable | Descripción |
|:--------:|:------------|
| C310 | Categoría ocupacional |
| C311 | Sector institucional |
| C312 | Registro en SUNAT |
| C313 | Lleva libros contables |
| C317 | Número de trabajadores en empresa |
| C317A | Número de trabajadores (complemento) |
| C335 | ¿Busca otro empleo? |
| C318_T | Horas en empleo principal |
| C328_T | Horas en empleo secundario |
| whoraT | Total horas trabajadas (variable calculada EPEN) |
| C331 | Horas que desea trabajar |
| SEGURO1 | Tipo de seguro de salud |
| C361_1 | Tiene EsSalud |
| C361_5 | Tiene SIS |
| C364_1 | Tiene AFP |
| C364_2 | Tiene SNP/ONP |

### Restricciones
- **NO usar:** `P209H`, `C333`, `C334`, `fa_son24`
- C333 y C334 son las horas habituales deseadas — son parte directa del cálculo del target y constituyen **leakage**
- No se realiza train/test split en este notebook

---
## 1. Cargar Librerías

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


---
## 2. Cargar Dataset

In [2]:
INPUT_PATH = Path('../data/feature_engineering/epen_fe_education.csv')

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f'No se encontró el archivo de entrada: {INPUT_PATH}\n'
        'Ejecuta primero: 04_feature_engineering/02_education_features.ipynb'
    )

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Dataset cargado : {INPUT_PATH.name}')
print(f'Dimensiones     : {df.shape[0]:,} filas x {df.shape[1]} columnas')

Dataset cargado : epen_fe_education.csv
Dimensiones     : 24,054 filas x 70 columnas


---
## 3. Validación Inicial

In [3]:
assert 'target_subempleo_horas' in df.columns, \
    "ERROR: 'target_subempleo_horas' no encontrado."
assert df['target_subempleo_horas'].isnull().sum() == 0, \
    'ERROR: target_subempleo_horas contiene nulos.'
print('target_subempleo_horas presente y sin nulos: OK')

for lv in ['P209H', 'C333', 'C334']:
    assert lv not in df.columns, f'ERROR: variable de leakage {lv} encontrada.'
print('Variables de leakage ausentes: OK')

print('\nDistribución del target:')
counts = df['target_subempleo_horas'].value_counts()
pct    = df['target_subempleo_horas'].value_counts(normalize=True) * 100
display(pd.DataFrame({'conteo': counts, 'porcentaje (%)': pct.round(2)}))

target_subempleo_horas presente y sin nulos: OK
Variables de leakage ausentes: OK

Distribución del target:


,conteo,porcentaje (%)
target_subempleo_horas,,
0,18064,75.1000
1,5990,24.9000


---
## 4. Crear Copia de Trabajo

In [4]:
df_fe = df.copy()
n_original = df_fe.shape[0]
features_created = []

print(f'Copia creada: {df_fe.shape[0]:,} filas x {df_fe.shape[1]} columnas')

Copia creada: 24,054 filas x 70 columnas


---
## 5. Categoría Ocupacional

**Variable base:** `C310`

| Código | Categoría |
|:------:|:----------|
| 1 | Empleador / patrono |
| 2 | Trabajador independiente |
| 3 | Empleado / obrero dependiente |
| 4 | Trabajador del hogar |
| 5-6 | Trabajador familiar no remunerado |

> Nota: Los códigos exactos pueden variar. Se verifica la presencia antes de crear cada variable.

In [5]:
if 'C310' in df_fe.columns:
    c310 = pd.to_numeric(df_fe['C310'], errors='coerce')

    df_fe['trabajador_independiente']         = (c310 == 2).astype(int)
    df_fe['trabajador_dependiente']           = (c310 == 3).astype(int)
    df_fe['empleador']                        = (c310 == 1).astype(int)
    df_fe['trabajador_hogar']                 = (c310 == 4).astype(int)
    df_fe['trabajador_familiar_no_remunerado']= c310.isin([5, 6, 8, 9, 10]).astype(int)

    nuevas_cat = [
        'trabajador_independiente', 'trabajador_dependiente', 'empleador',
        'trabajador_hogar', 'trabajador_familiar_no_remunerado'
    ]
    features_created += nuevas_cat

    print('Distribución de categoría ocupacional:')
    print(f'  Valores únicos en C310: {sorted(c310.dropna().unique().astype(int).tolist())}')
    for col in nuevas_cat:
        pct = df_fe[col].mean() * 100
        print(f'  {col:<40}: {df_fe[col].sum():>7,}  ({pct:.1f}%)')
else:
    print('ADVERTENCIA: C310 no encontrado.')

Distribución de categoría ocupacional:
  Valores únicos en C310: [1, 2, 3, 4, 6, 7, 8, 9]
  trabajador_independiente                :   7,477  (31.1%)
  trabajador_dependiente                  :  13,768  (57.2%)
  empleador                               :     934  (3.9%)
  trabajador_hogar                        :     674  (2.8%)
  trabajador_familiar_no_remunerado       :   1,109  (4.6%)


---
## 6. Sector de Empleo

**Variable base:** `C311`

| Código | Sector |
|:------:|:-------|
| 1 | Gobierno central |
| 2 | Gobierno regional / local |
| 3 | Empresa del Estado |
| 4 | Empresa mixta |
| 5 | Empresa privada |
| 6 | Organización sin fines de lucro |

**Features creados:** `sector_publico`, `sector_privado`

In [7]:
if 'C311' in df_fe.columns:
    c311 = pd.to_numeric(df_fe['C311'], errors='coerce')

    df_fe['sector_publico'] = c311.isin([1, 2, 3]).astype(int)
    df_fe['sector_privado'] = (c311 == 5).astype(int)
    features_created += ['sector_publico', 'sector_privado']

    print(f"sector_publico : {df_fe['sector_publico'].sum():>7,}  ({df_fe['sector_publico'].mean()*100:.1f}%)")
    print(f"sector_privado : {df_fe['sector_privado'].sum():>7,}  ({df_fe['sector_privado'].mean()*100:.1f}%)")
else:
    print('ADVERTENCIA: C311 no encontrado.')

sector_publico :   2,164  (9.0%)
sector_privado :  11,127  (46.3%)


---
## 7. Formalidad del Empleo

**Variables base:** `C312` (registro en SUNAT), `C313` (lleva libros contables)

| C312 | Registro |
|:----:|:---------|
| 1 | Sí, con RUC activo |
| 2 | Sí, RUC suspendido |
| 3 | No registrado |

**Features creados:** `empresa_registrada_sunat`, `empresa_no_registrada_sunat`, `lleva_libros_contables`, `posible_informalidad`

In [8]:
if 'C312' in df_fe.columns:
    c312 = pd.to_numeric(df_fe['C312'], errors='coerce')
    df_fe['empresa_registrada_sunat']    = c312.isin([1, 2]).astype(int)
    df_fe['empresa_no_registrada_sunat'] = (c312 == 3).astype(int)
    features_created += ['empresa_registrada_sunat', 'empresa_no_registrada_sunat']
    print(f"empresa_registrada_sunat    : {df_fe['empresa_registrada_sunat'].sum():,}  ({df_fe['empresa_registrada_sunat'].mean()*100:.1f}%)")
    print(f"empresa_no_registrada_sunat : {df_fe['empresa_no_registrada_sunat'].sum():,}  ({df_fe['empresa_no_registrada_sunat'].mean()*100:.1f}%)")
else:
    print('ADVERTENCIA: C312 no encontrado.')

if 'C313' in df_fe.columns:
    c313 = pd.to_numeric(df_fe['C313'], errors='coerce')
    df_fe['lleva_libros_contables'] = (c313 == 1).astype(int)
    features_created.append('lleva_libros_contables')
    print(f"lleva_libros_contables      : {df_fe['lleva_libros_contables'].sum():,}  ({df_fe['lleva_libros_contables'].mean()*100:.1f}%)")
else:
    print('ADVERTENCIA: C313 no encontrado.')

# Posible informalidad: no registrado en SUNAT y no lleva libros
cols_inf = []
if 'empresa_no_registrada_sunat' in df_fe.columns:
    cols_inf.append('empresa_no_registrada_sunat')
if 'lleva_libros_contables' in df_fe.columns:
    cols_inf.append('lleva_libros_contables')

if len(cols_inf) == 2:
    df_fe['posible_informalidad'] = (
        (df_fe['empresa_no_registrada_sunat'] == 1) & (df_fe['lleva_libros_contables'] == 0)
    ).astype(int)
    features_created.append('posible_informalidad')
    print(f"posible_informalidad        : {df_fe['posible_informalidad'].sum():,}  ({df_fe['posible_informalidad'].mean()*100:.1f}%)")

empresa_registrada_sunat    : 12,390  (51.5%)
empresa_no_registrada_sunat : 8,491  (35.3%)
lleva_libros_contables      : 1,037  (4.3%)
posible_informalidad        : 8,491  (35.3%)


---
## 8. Tamaño de Empresa

**Variable base:** `C317` (número de trabajadores en la empresa)

| Código | Rango |
|:------:|:------|
| 1 | 1 trabajador (unipersonal) |
| 2 | 2–10 trabajadores (microempresa) |
| 3 | 11–50 trabajadores (pequeña empresa) |
| 4 | 51–100 trabajadores (mediana empresa) |
| 5 | 101+ trabajadores (gran empresa) |

**Features creados:** `empresa_pequena`, `empresa_mediana_grande`

In [9]:
c317_col = 'C317' if 'C317' in df_fe.columns else ('C317A' if 'C317A' in df_fe.columns else None)

if c317_col:
    c317 = pd.to_numeric(df_fe[c317_col], errors='coerce')
    df_fe['empresa_pequena']        = c317.isin([1, 2]).astype(int)
    df_fe['empresa_mediana_grande'] = c317.isin([3, 4, 5]).astype(int)
    features_created += ['empresa_pequena', 'empresa_mediana_grande']

    print(f'Variable base usada: {c317_col}')
    print(f"empresa_pequena        : {df_fe['empresa_pequena'].sum():,}  ({df_fe['empresa_pequena'].mean()*100:.1f}%)")
    print(f"empresa_mediana_grande : {df_fe['empresa_mediana_grande'].sum():,}  ({df_fe['empresa_mediana_grande'].mean()*100:.1f}%)")
else:
    print('ADVERTENCIA: C317 y C317A no encontrados.')

Variable base usada: C317
empresa_pequena        : 17,473  (72.6%)
empresa_mediana_grande : 6,581  (27.4%)


---
## 9. Búsqueda de Otro Empleo

**Variable base:** `C335` (¿busca otro empleo o trabajo?)  
Codificación: 1 = Sí, 2 = No

**Feature creado:** `busca_otro_empleo`

In [10]:
if 'C335' in df_fe.columns:
    c335 = pd.to_numeric(df_fe['C335'], errors='coerce')
    df_fe['busca_otro_empleo'] = (c335 == 1).astype(int)
    features_created.append('busca_otro_empleo')

    pct = df_fe['busca_otro_empleo'].mean() * 100
    print(f"busca_otro_empleo : {df_fe['busca_otro_empleo'].sum():,}  ({pct:.1f}%)")

    print('\nTasa de subempleo segun busqueda de otro empleo:')
    display(df_fe.groupby('busca_otro_empleo')['target_subempleo_horas'].mean().rename('tasa_subempleo').to_frame())
else:
    print('ADVERTENCIA: C335 no encontrado.')

busca_otro_empleo : 2,365  (9.8%)

Tasa de subempleo segun busqueda de otro empleo:


,tasa_subempleo
busca_otro_empleo,
0,0.2140
1,0.5704


---
## 10. Variables de Horas Trabajadas

**Variables base:** `whoraT` (total horas EPEN), `C318_T` (horas principal), `C328_T` (horas secundario), `C331` (horas deseadas)

>  **Importante:** NO se usa `C333` ni `C334` (horas habituales deseadas — leakage directo del target).

**Features creados:** `horas_totales`, `horas_principal`, `horas_secundaria`, `trabaja_menos_35h`, `trabaja_mas_48h`, `sobrejornada`, `brecha_horas_normal`

In [11]:
# Horas totales: preferir whoraT (calculado por EPEN); si no, sumar principal + secundario
if 'whoraT' in df_fe.columns:
    df_fe['horas_totales'] = pd.to_numeric(df_fe['whoraT'], errors='coerce')
    print('Fuente de horas_totales: whoraT')
elif 'C318_T' in df_fe.columns and 'C328_T' in df_fe.columns:
    h_princ = pd.to_numeric(df_fe['C318_T'], errors='coerce').fillna(0)
    h_sec   = pd.to_numeric(df_fe['C328_T'], errors='coerce').fillna(0)
    df_fe['horas_totales'] = h_princ + h_sec
    print('Fuente de horas_totales: C318_T + C328_T')
elif 'C318_T' in df_fe.columns:
    df_fe['horas_totales'] = pd.to_numeric(df_fe['C318_T'], errors='coerce')
    print('Fuente de horas_totales: C318_T (solo principal)')
else:
    df_fe['horas_totales'] = np.nan
    print('ADVERTENCIA: No se encontró ninguna variable de horas.')

# Horas por empleo
if 'C318_T' in df_fe.columns:
    df_fe['horas_principal']  = pd.to_numeric(df_fe['C318_T'], errors='coerce')
    features_created.append('horas_principal')

if 'C328_T' in df_fe.columns:
    df_fe['horas_secundaria'] = pd.to_numeric(df_fe['C328_T'], errors='coerce').fillna(0)
    features_created.append('horas_secundaria')

features_created.append('horas_totales')

# Indicadores de jornada
df_fe['trabaja_menos_35h'] = (df_fe['horas_totales'] < 35).astype(int)
df_fe['trabaja_mas_48h']   = (df_fe['horas_totales'] > 48).astype(int)
df_fe['sobrejornada']      = (df_fe['horas_totales'] > 60).astype(int)
features_created += ['trabaja_menos_35h', 'trabaja_mas_48h', 'sobrejornada']

# Brecha respecto a horas normales deseadas (C331 = horas que quisiera trabajar)
if 'C331' in df_fe.columns:
    horas_deseadas = pd.to_numeric(df_fe['C331'], errors='coerce')
    df_fe['brecha_horas_normal'] = horas_deseadas - df_fe['horas_totales']
    features_created.append('brecha_horas_normal')
    print(f"\nbrecha_horas_normal — media: {df_fe['brecha_horas_normal'].mean():.2f} h")

print(f"\nhoras_totales (media)  : {df_fe['horas_totales'].mean():.2f} h")
print(f"trabaja_menos_35h     : {df_fe['trabaja_menos_35h'].sum():,}  ({df_fe['trabaja_menos_35h'].mean()*100:.1f}%)")
print(f"trabaja_mas_48h       : {df_fe['trabaja_mas_48h'].sum():,}  ({df_fe['trabaja_mas_48h'].mean()*100:.1f}%)")

Fuente de horas_totales: whoraT

brecha_horas_normal — media: -1.36 h

horas_totales (media)  : 44.50 h
trabaja_menos_35h     : 5,556  (23.1%)
trabaja_mas_48h       : 8,009  (33.3%)


---
## 11. Variables de Protección Social

**Variables base:** `SEGURO1`, `C361_1` (EsSalud), `C361_5` (SIS), `C364_1` (AFP), `C364_2` (ONP/SNP)

**Features creados:** `tiene_seguro_salud`, `tiene_essalud`, `tiene_sis`, `tiene_pension`, `tiene_afp`, `tiene_snp`, `proteccion_social_completa`

In [12]:
seg_features = []

if 'SEGURO1' in df_fe.columns:
    seguro = pd.to_numeric(df_fe['SEGURO1'], errors='coerce')
    df_fe['tiene_seguro_salud'] = seguro.isin([1, 2, 3, 4, 5]).astype(int)
    seg_features.append('tiene_seguro_salud')

if 'C361_1' in df_fe.columns:
    df_fe['tiene_essalud'] = (pd.to_numeric(df_fe['C361_1'], errors='coerce') == 1).astype(int)
    seg_features.append('tiene_essalud')

if 'C361_5' in df_fe.columns:
    df_fe['tiene_sis'] = (pd.to_numeric(df_fe['C361_5'], errors='coerce') == 1).astype(int)
    seg_features.append('tiene_sis')

if 'C364_1' in df_fe.columns:
    afp = (pd.to_numeric(df_fe['C364_1'], errors='coerce') == 1).astype(int)
    df_fe['tiene_afp'] = afp
    seg_features.append('tiene_afp')
else:
    afp = pd.Series(0, index=df_fe.index)

if 'C364_2' in df_fe.columns:
    snp = (pd.to_numeric(df_fe['C364_2'], errors='coerce') == 1).astype(int)
    df_fe['tiene_snp'] = snp
    seg_features.append('tiene_snp')
else:
    snp = pd.Series(0, index=df_fe.index)

df_fe['tiene_pension'] = ((afp == 1) | (snp == 1)).astype(int)
seg_features.append('tiene_pension')

# Protección social completa: tiene seguro de salud Y pensión
if 'tiene_seguro_salud' in df_fe.columns:
    df_fe['proteccion_social_completa'] = (
        (df_fe['tiene_seguro_salud'] == 1) & (df_fe['tiene_pension'] == 1)
    ).astype(int)
    seg_features.append('proteccion_social_completa')

features_created += seg_features

print('Variables de protección social:')
for col in seg_features:
    if col in df_fe.columns:
        pct = df_fe[col].mean() * 100
        print(f'  {col:<30}: {df_fe[col].sum():>7,}  ({pct:.1f}%)')

Variables de protección social:
  tiene_seguro_salud            :  22,288  (92.7%)
  tiene_essalud                 :  10,255  (42.6%)
  tiene_sis                     :  11,171  (46.4%)
  tiene_afp                     :   9,875  (41.1%)
  tiene_snp                     :   3,085  (12.8%)
  tiene_pension                 :  12,958  (53.9%)
  proteccion_social_completa    :  12,443  (51.7%)


---
## 12. Validaciones Finales

In [13]:
assert df_fe.shape[0] == n_original, \
    f'ERROR: el número de filas cambió. Original: {n_original}, actual: {df_fe.shape[0]}'
print(f'Número de filas sin cambios : OK ({n_original:,})')

assert 'target_subempleo_horas' in df_fe.columns
print('target_subempleo_horas presente: OK')

for lv in ['P209H', 'C333', 'C334']:
    assert lv not in df_fe.columns, f'ERROR: {lv} presente.'
print('Variables de leakage ausentes: OK')

print(f'\nFeatures de empleo creados ({len(features_created)}):')
for feat in features_created:
    nulos = df_fe[feat].isnull().sum()
    print(f'  {feat:<38}: {nulos:>6} nulos  ({nulos/len(df_fe)*100:.2f}%)')

print(f'\nDimensiones finales  : {df_fe.shape[0]:,} filas x {df_fe.shape[1]} columnas')
print(f'Columnas nuevas      : {df_fe.shape[1] - df.shape[1]}')

Número de filas sin cambios : OK (24,054)
target_subempleo_horas presente: OK
Variables de leakage ausentes: OK

Features de empleo creados (30):
  trabajador_independiente              :      0 nulos  (0.00%)
  trabajador_dependiente                :      0 nulos  (0.00%)
  empleador                             :      0 nulos  (0.00%)
  trabajador_hogar                      :      0 nulos  (0.00%)
  trabajador_familiar_no_remunerado     :      0 nulos  (0.00%)
  sector_publico                        :      0 nulos  (0.00%)
  sector_privado                        :      0 nulos  (0.00%)
  sector_publico                        :      0 nulos  (0.00%)
  sector_privado                        :      0 nulos  (0.00%)
  empresa_registrada_sunat              :      0 nulos  (0.00%)
  empresa_no_registrada_sunat           :      0 nulos  (0.00%)
  lleva_libros_contables                :      0 nulos  (0.00%)
  posible_informalidad                  :      0 nulos  (0.00%)
  empresa_pequena     

---
## 13. Guardar Resultados

In [14]:
OUTPUT_DIR = Path('../data/feature_engineering')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset con features de empleo
out_main = OUTPUT_DIR / 'epen_fe_employment.csv'
df_fe.to_csv(out_main, index=False)
print(f'Dataset guardado : {out_main}  ({df_fe.shape[0]:,} x {df_fe.shape[1]})')

# Reporte de features creados
report = pd.DataFrame({
    'feature'   : features_created,
    'tipo_dato' : [str(df_fe[f].dtype) for f in features_created],
    'nulos'     : [df_fe[f].isnull().sum() for f in features_created],
    'n_unicos'  : [df_fe[f].nunique() for f in features_created],
    'pct_nulos' : [round(df_fe[f].isnull().mean() * 100, 4) for f in features_created],
})
out_report = OUTPUT_DIR / 'employment_features_created.csv'
report.to_csv(out_report, index=False)
print(f'Reporte guardado : {out_report}  ({len(report)} features)')

print('\nTodos los archivos guardados correctamente.')
print('Siguiente paso -> 04_income_features.ipynb')

Dataset guardado : ..\data\feature_engineering\epen_fe_employment.csv  (24,054 x 98)
Reporte guardado : ..\data\feature_engineering\employment_features_created.csv  (30 features)

Todos los archivos guardados correctamente.
Siguiente paso -> 04_income_features.ipynb
